The notebook uses these default models:

```python
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
MODEL_NAME = "Qwen/Qwen3.5-4B-Base"

In [1]:
!python -m pip uninstall -y torch torchvision torchaudio flash-linear-attention fla-core
!python -m pip install -U pip
!python -m pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130
!python -m pip install -U transformers accelerate safetensors sentencepiece huggingface_hub einops
!python -m pip install -U flash-linear-attention
!python -m pip install -U requests beautifulsoup4 lxml
#!pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu126 #for CUDA version 12.6
#

Found existing installation: torch 2.12.0
Uninstalling torch-2.12.0:
  Successfully uninstalled torch-2.12.0
Found existing installation: flash-linear-attention 0.5.0
Uninstalling flash-linear-attention-0.5.0:
  Successfully uninstalled flash-linear-attention-0.5.0
Found existing installation: fla-core 0.5.0
Uninstalling fla-core-0.5.0:
  Successfully uninstalled fla-core-0.5.0
Looking in indexes: https://artifacts.dell.com/artifactory/api/pypi/python/simple, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-1002685-pypi-prd-local/simple
Looking in indexes: https://download.pytorch.org/whl/cu130, https://artifacts.dell.com/artifactory/api/pypi/ailfc-1003745-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aia-1001238-pypi-prd-local/simple, https://artifacts.dell.com/artifactory/api/pypi/aiops-10

# Chapter-01-AI-Workflows

In [2]:
import os, torch, transformers, re, json, requests
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import Optional
import html as html_lib
from bs4 import BeautifulSoup

usr_input = "new car model"

HF_CACHE_DIR = "./hf_cache"
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

OFFLINE_MODE = True

USE_LOCAL_SNAPSHOT_PATH = True
HF_TOKEN = None
MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 300
TEMPERATURE = 0.7
TOP_P = 0.9
FORCE_CPU = False
ATTN_IMPLEMENTATION = "eager"

cache_path = Path(HF_CACHE_DIR).expanduser().resolve()
cache_path.mkdir(parents=True, exist_ok=True)

os.environ["TORCH_CUDNN_SDPA_ENABLED"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

if OFFLINE_MODE:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Niektore kniznice citaju aj tieto premenne
os.environ["HF_HUB_CACHE"] = str(cache_path)
os.environ["TRANSFORMERS_CACHE"] = str(cache_path)

def find_local_snapshot(model_name: str, cache_dir: Path) -> Path | None:
    repo_folder = "models--" + model_name.replace("/", "--")
    snapshots_root = cache_dir / repo_folder / "snapshots"

    if not snapshots_root.exists():
        return None

    candidates = [
        path for path in snapshots_root.iterdir()
        if path.is_dir() and (path / "config.json").exists()]

    if not candidates:
        return None

    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

snapshot_path = None

if USE_LOCAL_SNAPSHOT_PATH:
    snapshot_path = find_local_snapshot(MODEL_NAME, cache_path)

if snapshot_path:
    LOAD_TARGET = str(snapshot_path)
    print(f"Using local snapshot:")
    print(LOAD_TARGET)
else:
    LOAD_TARGET = MODEL_NAME
    print(f"Using model name with cache_dir:")
    print(LOAD_TARGET)

print(f"\nCache dir: {cache_path}")
print(f"Transformers version: {transformers.__version__}")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

tokenizer_kwargs = {"trust_remote_code": True,"local_files_only": OFFLINE_MODE,"cache_dir": str(cache_path)}

model_kwargs = {"trust_remote_code": True,"local_files_only": OFFLINE_MODE,"cache_dir": str(cache_path),"attn_implementation": ATTN_IMPLEMENTATION}

if HF_TOKEN and not OFFLINE_MODE:
    tokenizer_kwargs["token"] = HF_TOKEN
    model_kwargs["token"] = HF_TOKEN

use_cuda = torch.cuda.is_available() and not FORCE_CPU

try:
    transformers_major = int(transformers.__version__.split(".")[0])
except Exception:
    transformers_major = 4

dtype_key = "dtype" if transformers_major >= 5 else "torch_dtype"

if use_cuda:
    model_kwargs[dtype_key] = torch.float16
    model_kwargs["device_map"] = "auto"
    print("\nCUDA detected. Loading model on GPU with eager attention.")
else:
    model_kwargs[dtype_key] = torch.float32
    print("\nCUDA disabled or not detected. Loading model on CPU.")

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LOAD_TARGET,**tokenizer_kwargs)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(LOAD_TARGET,**model_kwargs)

if not use_cuda:
    model = model.to("cpu")

model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nModel loaded locally.")

Using local snapshot:
/home/dell/AI Agents Training/hf_cache/models--Qwen--Qwen3.5-0.8B/snapshots/2fc06364715b967f1860aea9cf38778875588b17

Cache dir: /home/dell/AI Agents Training/hf_cache
Transformers version: 5.9.0
Torch version: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA H100 80GB HBM3 MIG 1g.10gb

CUDA detected. Loading model on GPU with eager attention.

Loading tokenizer...
Loading model...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


Model loaded locally.


In [3]:
### 1. Basic prompting with role

In [4]:
if "tokenizer" not in globals() or "model" not in globals():
    raise RuntimeError("Run model load")

def run_local_llm(
    prompt: str,
    system_prompt: Optional[str] = None,
    max_new_tokens: Optional[int] = None,
    temperature: Optional[float] = None,
    top_p: Optional[float] = None,
) -> str:
    prompt = str(prompt).strip()
    if not prompt:
        return ""

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": str(system_prompt).strip()})
    messages.append({"role": "user", "content": prompt})

    if getattr(tokenizer, "chat_template", None):
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        if system_prompt:
            input_text = f"System:\n{system_prompt.strip()}\n\nUser:\n{prompt}\n\nAssistant:\n"
        else:
            input_text = prompt

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    gen_temperature = TEMPERATURE if temperature is None else temperature
    gen_top_p = TOP_P if top_p is None else top_p
    gen_max_new_tokens = MAX_NEW_TOKENS if max_new_tokens is None else max_new_tokens

    generation_kwargs = {
        "max_new_tokens": gen_max_new_tokens,
        "do_sample": gen_temperature > 0,
        "pad_token_id": tokenizer.pad_token_id,
        "use_cache": True,
    }

    if tokenizer.eos_token_id is not None:
        generation_kwargs["eos_token_id"] = tokenizer.eos_token_id

    if gen_temperature > 0:
        generation_kwargs["temperature"] = gen_temperature
        generation_kwargs["top_p"] = gen_top_p

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            **generation_kwargs,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()


def generate_x_post(topic: str) -> str:
    system_prompt = """
                        You are an expert social media manager.
                        You write concise, clear and engaging posts for X, formerly Twitter.
                    """

    prompt = f"""
                Generate one short X post for this topic:

                <topic>
                {topic}
                </topic>

                Requirements:
                - keep it concise and impactful
                - avoid hashtags
                - use at most one emoji
                - use clean line breaks
                - return only the post, no explanation
                """

    return run_local_llm(prompt,system_prompt=system_prompt,max_new_tokens=180,temperature=0.7,top_p=0.9)


#usr_input = "new car model"
x_post = generate_x_post(usr_input)

print("\nGenerated X post:")
print(x_post)



Generated X post:
New car model arrives! 🚗✨


In [5]:
if "run_local_llm" not in globals():
    raise RuntimeError("Run model load")

def generate_x_post(topic: str) -> str:
    system_prompt = """
                        You are an expert social media manager.
                        You write concise, readable and engaging posts for X, formerly Twitter.
                    """

    prompt = f"""
                Create a post for X based on the topic below.

                <topic>
                {topic}
                </topic>

            Rules:
            - concise and focused
            - no hashtags
            - maximum one emoji
            - clean formatting with line breaks
            - return only the final post
            """

    return run_local_llm(prompt,system_prompt=system_prompt,max_new_tokens=180,temperature=0.7,top_p=0.9)

def main():
    # usr_input = input("What should the post be about? ")
    #usr_input = "new car model"

    x_post = generate_x_post(usr_input)

    print("\nGenerated X post:")
    print(x_post)

### 02 More fitted prompt

In [6]:
if "run_local_llm" not in globals():
    raise RuntimeError("Run model load")

def generate_x_post(topic: str) -> str:
    system_prompt = """
                    You are an expert social media manager.
                    You write concise, readable and engaging posts for X, formerly Twitter.
                    """

    prompt = f"""
                Create a post for X based on the topic below.

                <topic>
                {topic}
                </topic>

            Rules:
            - concise and focused
            - no hashtags
            - maximum one emoji
            - clean formatting with line breaks
            - return only the final post
            """

    return run_local_llm(prompt,system_prompt=system_prompt,max_new_tokens=180,temperature=0.7,top_p=0.9)

def main():
    # usr_input = input("What should the post be about? ")
    #usr_input = "new car model"

    x_post = generate_x_post(usr_input)

    print("\nGenerated X post:")
    print(x_post)


if __name__ == "__main__":
    main()



Generated X post:
🚗 New car model launch! 🚗🔥

The latest generation is here. What's your favorite feature?


### 03 Few shot prompting

In [7]:
if "run_local_llm" not in globals():
    raise RuntimeError("Run model load")

def load_post_examples(path: str = "post-examples.json") -> list[dict]:

    examples_path = Path(path)

    if examples_path.exists():
        with examples_path.open("r", encoding="utf-8") as f:
            return json.load(f)

    print(f"Warning: {path} sa nenasiel. Pouzivam fallback few-shot priklady.")
    return [{
                    "topic": "electric city car",
                    "post": "Small car. Big city energy.\n\nBuilt for short trips, tight streets, and everyday freedom."
                },
                {
                    "topic": "new laptop launch",
                    "post": "A laptop should feel fast before you even think about specs.\n\nThis one is built for smooth work, clean focus, and fewer compromises."
                },
                {
                    "topic": "coffee shop opening",
                    "post": "New place, fresh coffee, warm lights.\n\nThe kind of corner where one quick espresso somehow turns into an hour."
                }]


def build_examples_block(examples: list[dict]) -> str:
    examples_str = ""

    for i, example in enumerate(examples, 1):
        examples_str += f"""
        <example-{i}>
            <topic>
            {example.get("topic", "")}
            </topic>

            <generated-post>
            {example.get("post", "")}
            </generated-post>
        </example-{i}>
        """

    return examples_str.strip()


def generate_x_post(topic: str) -> str:
    examples = load_post_examples()
    examples_str = build_examples_block(examples)

    system_prompt = """
                        You are an expert social media manager.
                        You write concise, readable and engaging posts for X, formerly Twitter.
                        Follow the style of the examples, but do not copy their content.
                    """

    prompt = f"""
                Generate one X post for this topic:

                <topic>
                {topic}
                </topic>

                Use these examples only as style guidance:

                <examples>
                {examples_str}
                </examples>

                Rules:
                - do not copy phrases from the examples
                - no hashtags
                - maximum one emoji
                - clean line breaks
                - return only the final post
                """

    return run_local_llm(
        prompt,system_prompt=system_prompt,max_new_tokens=200,temperature=0.7,top_p=0.9)


def main():
    # usr_input = input("What should the post be about? ")
    #usr_input = "new car model"

    x_post = generate_x_post(usr_input)

    print("\nGenerated X post:")
    print(x_post)


if __name__ == "__main__":
    main()



Generated X post:
New tech models aren't just new cars; they are new ways of thinking.

Don't wait for a release to start building.

Start small.
Focus on the problem you can fix today.
Move fast.


### 04 Multi step prompting with web scrapper

In [8]:
if "run_local_llm" not in globals():
    raise RuntimeError("Run model load")

def get_website_html(url: str) -> str:
    try:
        response = requests.get(
            url,
            timeout=30,
            headers={
                "User-Agent": (
                                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                "AppleWebKit/537.36 (KHTML, like Gecko) "
                                "Chrome/120.0 Safari/537.36"
                                ),"Accept-Language": "sk-SK,sk;q=0.9,en;q=0.8"})
        response.raise_for_status()
        return response.text
    except requests.RequestException as e:
        print(f"Error fetching the URL {url}: {e}")
        return ""

def normalize_text(text: str) -> str:

    text = html_lib.unescape(str(text))
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def deduplicate_lines(text: str) -> str:
    seen = set()
    lines = []

    for raw_line in normalize_text(text).splitlines():
        line = normalize_text(raw_line)

        if not line:
            continue

        key = line.lower()

        if key in seen:
            continue

        seen.add(key)
        lines.append(line)

    return "\n".join(lines).strip()


def remove_noise_from_soup(soup: BeautifulSoup) -> None:
    for tag in soup.find_all([
                                "script",
                                "style",
                                "noscript",
                                "svg",
                                "canvas",
                                "iframe",
                                "form",
                                "input",
                                "button",
                                "select",
                                "textarea",
                                "nav",
                                "header",
                                "footer",
                                "aside",
                            ]):tag.decompose()

    # Bezpecnejsie selektory pre bezny webovy sum.
    noisy_selectors = [
                        "[aria-hidden='true']",
                        "[role='navigation']",
                        "[role='banner']",
                        "[role='contentinfo']",
                        ".cookie",
                        ".cookies",
                        "#cookie",
                        "#cookies",
                        ".cookie-banner",
                        ".cookie-consent",
                        ".modal",
                        ".popup",
                        ".newsletter",
                        ".breadcrumb",
                        ".breadcrumbs",
                        ".menu",
                        ".navbar",
                        ".navigation",
                        ".site-header",
                        ".site-footer",
                        ".footer",
                        ".header",
                        ".social",
                        ".share"
                        ]

    for selector in noisy_selectors:
        for tag in soup.select(selector):
            tag.decompose()


def pick_main_container(soup: BeautifulSoup):
    return (
        soup.find("main")
        or soup.find("article")
        or soup.select_one("[role='main']")
        or soup.select_one(".main")
        or soup.select_one("#main")
        or soup.select_one(".content")
        or soup.select_one("#content")
        or soup.body
        or soup
    )


def extract_text_with_beautifulsoup(html: str, min_line_len: int = 3) -> str:
    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.get_text(" ", strip=True) if soup.title else ""

    meta_description = ""
    meta_tag = soup.find("meta", attrs={"name": re.compile(r"^description$", re.I)})
    if meta_tag and meta_tag.get("content"):
        meta_description = meta_tag["content"].strip()

    remove_noise_from_soup(soup)
    main_container = pick_main_container(soup)

    parts = []

    if title:
        parts.append(title)

    if meta_description:
        parts.append(meta_description)

    text_tags = main_container.find_all(["h1", "h2", "h3", "h4", "p", "li"], recursive=True)

    for tag in text_tags:
        text = normalize_text(tag.get_text(" ", strip=True))

        if len(text) < min_line_len:
            continue

        if text.lower() in {"menu", "search", "login", "register", "privacy policy", "terms"}:
            continue

        parts.append(text)

    if not parts:
        parts.append(main_container.get_text("\n", strip=True))

    return deduplicate_lines("\n".join(parts))


def fit_for_prompt(text: str, max_chars: int = 12000) -> str:
    text = str(text).strip()

    if len(text) <= max_chars:
        return text

    head = text[: max_chars // 2]
    tail = text[-max_chars // 2 :]

    return head + "\n\n...[content shortened for local context window]...\n\n" + tail


def extract_core_website_content(html: str) -> str:
    content = extract_text_with_beautifulsoup(html)
    content = fit_for_prompt(content, max_chars=12000)

    if len(content) < 300:
        print(
                "Warning: Extracted text is too short. "
                "Web is rendered via JavaScript or blocking requests."
            )

    return content


def summarize_content(content: str) -> str:
    content = fit_for_prompt(content, max_chars=10000)

    system_prompt = """
                        You are an expert summarizer.
                        You create short, structured summaries.
                    """

    prompt = f"""
                Summarize the content below.

                <content>
                {content}
                </content>

                Requirements:
                - use bullet points
                - focus on the main facts
                - avoid unnecessary explanation
                - return only the summary
                """

    return run_local_llm(prompt,system_prompt=system_prompt,max_new_tokens=500,temperature=0.2,top_p=0.9)

def load_post_examples(path: str = "post-examples.json") -> list[dict]:
    examples_path = Path(path)

    if examples_path.exists():
        with examples_path.open("r", encoding="utf-8") as f:
            return json.load(f)

    print(f"Warning: {path} sa nenasiel. Pouzivam fallback few-shot priklady.")
    return [
                {
                    "topic": "electric city car",
                    "post": "Small car. Big city energy.\n\nBuilt for short trips, tight streets, and everyday freedom."
                },
                {
                    "topic": "new laptop launch",
                    "post": "A laptop should feel fast before you even think about specs.\n\nThis one is built for smooth work, clean focus, and fewer compromises."
                },
                {
                    "topic": "coffee shop opening",
                    "post": "New place, fresh coffee, warm lights.\n\nThe kind of corner where one quick espresso somehow turns into an hour."
            }]

def build_examples_block(examples: list[dict]) -> str:
    examples_str = ""

    for i, example in enumerate(examples, 1):
        examples_str += f"""
        <example-{i}>
            <topic>
            {example.get("topic", "")}
            </topic>

            <generated-post>
            {example.get("post", "")}
            </generated-post>
        </example-{i}>
        """

    return examples_str.strip()


def generate_x_post(summary: str) -> str:
    examples = load_post_examples()
    examples_str = build_examples_block(examples)

    system_prompt = """
                        You are an expert social media manager.
                        You write concise, readable and engaging posts for X, formerly Twitter.
                    """

    prompt = f"""
                Generate one X post based on this summary:

                <summary>
                {summary}
                </summary>

                Use these examples only as style guidance:

                <examples>
                {examples_str}
                </examples>

            Rules:
            - do not copy phrases from the examples
            - no hashtags
            - maximum one emoji
            - clean line breaks
            - return only the final post
            """

    return run_local_llm(prompt,system_prompt=system_prompt,max_new_tokens=220,temperature=0.7,top_p=0.9)


def main():
    website_url = "https://www.mazdausa.com/vehicles/mazda3-sedan"

    print("Fetching website HTML")
    html_content = get_website_html(website_url)

    if not html_content:
        print("Failed to fetch the website content. Exiting.")
        return

    print("Extracting core content with BeautifulSoup")
    core_content = extract_core_website_content(html_content)
    print("Extracted core content:")
    print(core_content)

    print("Summarizing the core content locally")
    summary = summarize_content(core_content)
    print("Generated summary:")
    print(summary)

    print("Generating X post locally")
    x_post = generate_x_post(summary)
    print("Generated X post:")
    print(x_post)


if __name__ == "__main__":
    main()


Fetching website HTML...
Error fetching the URL https://www.mazdausa.com/vehicles/mazda3-sedan: 503 Server Error: Service Unavailable for url: https://www.mazdausa.com/vehicles/mazda3-sedan
Failed to fetch the website content. Exiting.
